# Real time datacheck

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path().resolve().parent  # if notebook is inside /notebooks
DB_PATH = PROJECT_ROOT / "warehouse" / "trafiklab_realtime.duckdb"

print("DB_PATH:", DB_PATH)
con = duckdb.connect(str(DB_PATH))


DB_PATH: C:\Users\mdnrl\Documents\github\Ex_Jobb_Data_pipeline_for_Stockholm_Traffic_-_Transit\warehouse\trafiklab_realtime.duckdb


In [2]:
schemas = con.execute("""
    SELECT schema_name
    FROM information_schema.schemata
    ORDER BY schema_name
""").df()

schemas


,schema_name
0,analytics
1,information_schema
2,main
3,main
4,main
5,pg_catalog
6,raw_trafiklab


### List all tabels

In [3]:
tables = con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type='BASE TABLE'
    ORDER BY table_schema, table_name
""").df()

tables


,table_schema,table_name
0,raw_trafiklab,_dlt_loads
1,raw_trafiklab,_dlt_pipeline_state
2,raw_trafiklab,_dlt_version
3,raw_trafiklab,trafiklab_departures


In [4]:
def count_table(schema, table):
    return con.execute(f"SELECT COUNT(*) AS rows FROM {schema}.{table}").df()

# Edit these if your schema/table names differ
count_table("raw_trafiklab", "trafiklab_departures")


,rows
0,96


#### Preview raw data

In [6]:
df_raw = con.execute("""
    SELECT *
    FROM raw_trafiklab.trafiklab_departures
    ORDER BY response_timestamp DESC
    LIMIT 20
""").df()

df_raw


,response_timestamp,query_time,query_area_id,scheduled_time,realtime_time,delay_seconds,canceled,is_realtime,route_name,route_designation,...,agency__id,agency__name,agency__operator,stop__id,stop__name,stop__lat,stop__lon,route__name,scheduled_platform,realtime_platform
0,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:07:00+01:00,2026-01-19 18:07:00+01:00,0,False,False,None,103,...,None,None,None,None,None,NaN,NaN,None,None,None
1,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:10:00+01:00,2026-01-19 18:10:00+01:00,0,False,False,None,443,...,None,None,None,None,None,NaN,NaN,None,None,None
2,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:14:00+01:00,2026-01-19 18:14:00+01:00,0,False,False,None,183,...,None,None,None,None,None,NaN,NaN,None,None,None
3,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:21:00+01:00,2026-01-19 18:21:00+01:00,0,False,False,None,582,...,None,None,None,None,None,NaN,NaN,None,None,None
4,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:22:00+01:00,2026-01-19 18:22:00+01:00,0,False,False,None,543,...,None,None,None,None,None,NaN,NaN,None,None,None
5,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:28:00+01:00,2026-01-19 18:28:00+01:00,0,False,False,None,345,...,None,None,None,None,None,NaN,NaN,None,None,None
6,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:32:00+01:00,2026-01-19 18:32:00+01:00,0,False,False,None,282,...,None,None,None,None,None,NaN,NaN,None,None,None
7,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:37:00+01:00,2026-01-19 18:37:00+01:00,0,False,False,None,279,...,None,None,None,None,None,NaN,NaN,None,None,None
8,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:43:00+01:00,2026-01-19 18:43:00+01:00,0,False,False,None,133,...,None,None,None,None,None,NaN,NaN,None,None,None
9,2026-01-19 17:53:09+01:00,2026-01-19 17:53:00+01:00,740000001,2026-01-19 18:44:00+01:00,2026-01-19 18:44:00+01:00,0,False,False,None,48,...,None,None,None,None,None,NaN,NaN,None,None,None


In [7]:
df_fct = con.execute("""
    SELECT *
    FROM analytics.fct_departure_delays
    ORDER BY service_date DESC, hour_of_day DESC
    LIMIT 50
""").df()

df_fct

,departure_sk,scheduled_time,realtime_time,delay_seconds,route_designation,route_transport_mode,stop_name,service_date,hour_of_day,day_of_week,is_delayed
0,680025024922000001-2026-01-19 18:46:00+01-1,2026-01-19 18:46:00+01:00,2026-01-19 18:46:00+01:00,0,None,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
1,740048049773000001-2026-01-19 18:44:00+01-1,2026-01-19 18:44:00+01:00,2026-01-19 18:44:00+01:00,0,48,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
2,740133031596000001-2026-01-19 18:43:00+01-1,2026-01-19 18:43:00+01:00,2026-01-19 18:43:00+01:00,0,133,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
3,740279014219000001-2026-01-19 18:37:00+01-1,2026-01-19 18:37:00+01:00,2026-01-19 18:37:00+01:00,0,279,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
4,740282025562000001-2026-01-19 18:32:00+01-1,2026-01-19 18:32:00+01:00,2026-01-19 18:32:00+01:00,0,282,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
5,740345011869000001-2026-01-19 18:28:00+01-1,2026-01-19 18:28:00+01:00,2026-01-19 18:28:00+01:00,0,345,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
6,740543055639000001-2026-01-19 18:22:00+01-1,2026-01-19 18:22:00+01:00,2026-01-19 18:22:00+01:00,0,543,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
7,740582038998000001-2026-01-19 18:21:00+01-1,2026-01-19 18:21:00+01:00,2026-01-19 18:21:00+01:00,0,582,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
8,740183003919000001-2026-01-19 18:14:00+01-1,2026-01-19 18:14:00+01:00,2026-01-19 18:14:00+01:00,0,183,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0
9,740443033099000001-2026-01-19 18:10:00+01-1,2026-01-19 18:10:00+01:00,2026-01-19 18:10:00+01:00,0,443,TRAIN,Stockholm Centralstation,2026-01-19,18,1,0


### Delays

In [8]:
con.execute("""
    SELECT
        route_transport_mode,
        COUNT(*) AS n,
        AVG(delay_seconds) AS avg_delay
    FROM analytics.fct_departure_delays
    GROUP BY 1
    ORDER BY n DESC
""").df()


,route_transport_mode,n,avg_delay
0,TRAIN,96,0.0
